# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# 12 · Does the direct-state feature gain persist over time?

**Feature research remains open.** This milestone validates the existing 62 direct observed-state fields, not another untested feature bank. Round 4 improved 0.683160 → 0.667623 on one reused fold, but its adjusted upper difference bound is only −0.000010887. Geometry failed. Do not rerun that treatment or the first-fold models.

The three historical figures below use your actual uploaded Round 4 aggregates. New figures require your local outputs. No synthetic results are substituted.

In [ ]:
from pathlib import Path
import json, os, sys, subprocess, signal
import plotly.io as pio
KIT = Path('/home/sagemaker-user/nfl_feature_round5')
OUT = Path('/home/sagemaker-user/nfl-feature-round5-results')
PY = Path('/home/sagemaker-user/nfl-player-trajectory/.venv/bin/python')
if not KIT.is_dir() or not PY.is_file():
    raise FileNotFoundError('Use the existing NFL space and upload/extract the Round 5 package first.')
sys.path.insert(0, str(KIT))
import visuals
pio.renderers.default = 'plotly_mimetype'
def run(stage, fold=None):
    command = [str(PY), str(KIT/'run_round.py'), stage]
    if fold is not None: command += ['--fold', str(fold)]
    env = os.environ.copy()
    env.update({'OMP_NUM_THREADS':'2','OPENBLAS_NUM_THREADS':'2','PYTHONDONTWRITEBYTECODE':'1'})
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, env=env)
    try:
        for line in process.stdout: print(line, end='')
        code = process.wait()
    except KeyboardInterrupt:
        process.send_signal(signal.SIGINT)
        try: process.wait(timeout=12)
        except subprocess.TimeoutExpired: process.terminate()
        raise
    if code:
        raise RuntimeError(f'{stage} stopped (exit {code}). Preserve checkpoints and export the report. Do not alter settings.')
def show(fig, name):
    visuals.save(fig, OUT, name).show()
def receipt(name):
    return visuals.load(OUT/name)


In [ ]:
show(visuals.discovery_scores(KIT),'round4_scores')
show(visuals.discovery_intervals(KIT),'round4_intervals')
show(visuals.discovery_horizon(KIT),'round4_horizon')

## 1. Verify the original models and frozen folds

Four original coordinate models are forward-replayed without refitting. The same 1,024-play selection and same chronological game windows remain fixed. This is **temporal replication on previously used game splits**, not untouched validation or cross-season testing. Offline cached runtime only; no package installation. Stop if preflight fails.

In [ ]:
run('preflight')
p = receipt('preflight.json')
assert p['status'] == 'temporal_preflight_passed'
print(json.dumps(p['parent_replay'], indent=2))
show(visuals.fold_populations(OUT),'fold_populations')

## 2. Rebuild 32 training-play examples and verify exact parity

This smoke independently rebuilds raw observed-state fields even when the Round 4 cache exists. Legacy features use their historical float32/float64 conversion followed by exact equality. No targets are read from raw output CSVs.

In [ ]:
run('smoke')
assert receipt('smoke.json')['status'] == 'temporal_state_smoke_passed'
receipt('smoke.json')

## 3. Prepare only missing state checkpoints

The original 699 Round 4 per-play state files are reused read-only. The remaining selected plays are prepared under the same feature definition. Previously completed Round 5 files also resume by hash. No goal-geometry feature is fitted.

In [ ]:
run('prepare')
assert receipt('preparation.json')['status'] == 'temporal_state_ready'
show(visuals.cache_reuse(OUT),'checkpoint_reuse')
receipt('preparation.json')

## Next bounded step

Save this notebook. Open **13_direct_state_temporal_ablation.ipynb** only after preparation passes. Keep previous source and results directories unchanged. No additional data download, first-fold fit, feature tuning, or GitHub publication occurs here.